# Boltz-1 MHC-I fine-tune on a free cloud T4

TACC access did not come through, so this is the fallback: a **16 GB T4**, which is
what both Google Colab's free tier and Kaggle's free tier hand out. The laptop
4060 has 8 GB and OOMs inside triangular attention even with the trunk frozen
(`reports/GPU_REQUIREMENTS.md`). 16 GB is the difference between that and a run.

This notebook works unchanged on **Colab** and on **Kaggle**. It detects which one
it is on and adjusts paths, storage and the resume story.

**Kaggle is the better host for this**, and it is worth knowing why before you pick:

| | Colab free | Kaggle free |
|---|---|---|
| GPU | T4 16 GB, *when available* | **T4 x2** (32 GB total, one used) |
| Session cap | 12 h, idle-disconnects at ~90 min | 12 h |
| Weekly budget | undocumented, ~15-30 h, varies with usage history | **30 h, stated** |
| Runs with the tab closed | no | **yes** -- "Save & Run All (Commit)" |
| Persistence | your Google Drive | Kaggle Datasets / notebook output |

The last two rows are what matter for a job measured in hours. On Colab you must
babysit the tab; on Kaggle you commit and walk away.

On Kaggle choose the **`GPU T4 x2`** accelerator. (The P100 was retired on
2026-09-15 and is no longer offered; older guidance about avoiding it -- Pascal,
sm_60, which recent PyTorch wheels no longer build kernels for -- is now moot.)
This notebook uses one of the two T4s, and `trainer.devices: 1` keeps Lightning
on a single GPU; sharding across both is real work (DeepSpeed/FSDP) and not where
the next result comes from.

### Before you start

Run `python src/make_cloud_bundle.py` **on the laptop** once. It produces a
~575 MB `mhc1-data-bundle.tar.gz` holding `data/processed/` and `data/msa/` --
the output of milestone 1, and the only part of `data/` that a cloud session
cannot cheaply regenerate. Then:

* **Colab** -- upload it to Drive, e.g. `MyDrive/mhc1/mhc1-data-bundle.tar.gz`.
* **Kaggle** -- create a Dataset from it and attach that dataset to this notebook.

The 3.8 GB of `data/assets/` is *not* in the bundle. This notebook re-downloads
the checkpoint and the symmetry pickle from their original hosts, which is faster
than pushing them through Drive.

## 1. What did we actually get?

In [ ]:
# Identify the host and the card before doing anything expensive.
import os, subprocess, sys, textwrap, shutil

ON_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

PLATFORM = "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "local"
print(f"platform: {PLATFORM}")

# Session clock. Both platforms cap a SESSION at 12 h, but `max_time` in the
# config counts from when the TRAINER starts -- and everything before it (asset
# download, clone, probes, a sweep, a failed run or three) comes out of the same
# 12 h. Sizing max_time as if the two coincided is how a session gets killed
# mid-epoch, which is exactly the overrun case where Kaggle's output upload is
# best-effort. Section 8 uses this to compute what is actually left.
import time as _time
SESSION_START = _time.time()
SESSION_CAP_H = 12.0

import torch
if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. Colab: Runtime > Change runtime type > T4 GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2."
    )

name = torch.cuda.get_device_name(0)
cap  = torch.cuda.get_device_capability(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"gpu:      {name}  (sm_{cap[0]}{cap[1]}, {vram:.1f} GiB)")
print(f"torch:    {torch.__version__}")

# The two failure modes worth catching now rather than 20 minutes in.
if cap[0] < 7:
    print(textwrap.dedent("""
        !! Pre-Volta card (sm_6x). Recent torch wheels ship no kernels for it,
        !! so the run will die with 'no kernel image is available'.
    """))
if vram < 14:
    print(f"!! Only {vram:.1f} GiB. The budget in reports/CLOUD_GPU.md assumes ~16.")

print(f"cpus:     {os.cpu_count()}")
# '/' is the container root -- neither /kaggle/working (20 GiB cap) nor
# /kaggle/tmp (~60 GiB scratch), which are the two that actually bind.
for _lbl, _path in ([("working", "/kaggle/working"), ("tmp", "/kaggle/tmp")]
                    if ON_KAGGLE else [("disk", "/")]):
    try:
        print(f"{_lbl + ':':<10}{shutil.disk_usage(_path).free / 1024**3:.0f} GiB free")
    except OSError:
        print(f"{_lbl + ':':<10}not present yet")

# Internet is OFF by default on Kaggle and this notebook cannot work without it:
# it clones from GitHub and pulls 3.8 GB of assets. Catch it here, not in cell 4.
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=8).close()
    print("internet:  ok")
except OSError:
    print(
        "\n!! NO INTERNET. On Kaggle this is off by default.\n"
        "!!   right sidebar > Notebook options > Internet > On\n"
        "!! It requires a phone-verified account (Settings > Phone Verification).\n"
        "!! If the toggle is missing right after verifying, create a fresh\n"
        "!! notebook -- the option appears on new ones first."
    )

## 2. Persistent storage

Everything under the session's own filesystem dies when the session does. The run
directory -- checkpoints, logs, `val_state.json` -- has to live somewhere that
survives, or a 12 h cap means starting over every time.

In [ ]:
from pathlib import Path
import glob, json, tarfile, time

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/mhc1-runs/t4")
    WORK    = Path("/content")
elif ON_KAGGLE:
    # /kaggle/working is capped at 20 GiB and is also what gets saved as the
    # version's output. The repo carries 3.8 GB of re-downloadable assets plus a
    # boltz-src checkout -- a fifth of that budget, saved pointlessly on every
    # commit, and competing for space with ~8 GB of checkpoints. /kaggle/tmp has
    # ~60 GiB and is not persisted, which is exactly right for things that can be
    # fetched again.
    RUN_DIR = Path("/kaggle/working/runs/t4")     # persisted, counts to 20 GiB
    WORK    = Path("/kaggle/tmp")                 # scratch, ~60 GiB, not saved
else:
    RUN_DIR = Path.home() / "mhc1-runs" / "t4"
    WORK    = Path.home()

RUN_DIR.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)   # /kaggle/tmp does not exist by default
REPO = WORK / "mhc1-boltz"
print(f"run dir: {RUN_DIR}")
print(f"repo:    {REPO}")

# A ~4 GB checkpoint lands here when val/lddt improves, plus last.ckpt.
# Free Drive is 15 GB total, so check before rather than after.
if ON_COLAB:
    free = shutil.disk_usage(RUN_DIR).free / 1024**3
    print(f"drive:   {free:.1f} GiB free  (need ~10 GiB: best + last + headroom)")
elif ON_KAGGLE:
    used = sum(f.stat().st_size for f in Path("/kaggle/working").rglob("*")
               if f.is_file()) / 1024**3
    print(f"working: {used:.1f} / 20.0 GiB used  "
          f"(a checkpoint is ~4 GB; best + last is ~8)")

## 3. Code: repo, upstream Boltz, and the two-hunk patch

In [ ]:
# The repo does not vendor boltz-src/ -- it is a clean v1.0.0 tree plus our patch,
# so this reproduces it exactly. See reports/UPSTREAM_PATCHES.md.
def run(cmd, **kw):
    # A bare subprocess.run inherits the kernel's OS-level fd 1/2, which ipykernel
    # does NOT redirect -- it only swaps the sys.stdout object. So the child's
    # output goes to the kernel log, invisible in the cell, and a failure would
    # report only the command string with the real error lost. Stream it.
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, **kw)
    tail = []
    for line in proc.stdout:
        print("   " + line.rstrip()[:200], flush=True)
        tail.append(line)
        del tail[:-40]
    proc.wait()
    if proc.returncode:
        raise SystemExit(f"failed ({proc.returncode}): {cmd}\n" + "".join(tail))
    return proc

if not REPO.exists():
    run(f"git clone --depth 1 https://github.com/SaifSyed08/mhc1-boltz.git {REPO}")
else:
    print(f"{REPO} exists, leaving it alone")

BOLTZ = REPO / "boltz-src"
if not BOLTZ.exists():
    run(f"git clone https://github.com/jwohlwend/boltz.git {BOLTZ}")
    run(f"git -C {BOLTZ} checkout v1.0.0")
    run(f"git -C {BOLTZ} apply ../patches/boltz-v1.0.0-training-path.patch")
    print("patch applied")
else:
    print("boltz-src exists, leaving it alone")

# Verify, on BOTH branches. A directory-existence check is not a patch check: if a
# previous attempt cloned and then failed to apply, the directory exists and every
# later run would silently train on unpatched Boltz -- which is exactly how the
# cloud once ended up with a worse Boltz than the laptop. Sentinels, one per file
# that matters on the training path.
SENTINELS = {
    "src/boltz/model/modules/trunk.py":          "BOLTZ_CHUNK_IN_TRAINING",
    "src/boltz/model/modules/diffusion.py":      "center_random_augmentation",
    "src/boltz/data/feature/featurizer.py":      "chain_constraint_features = {}",
    "src/boltz/model/optim/scheduler.py":        "except TypeError",
    "scripts/train/train.py":                    "ckpt_every_minutes",
    "src/boltz/model/model.py":                  "oom-context",
}
missing = [f for f, tok in SENTINELS.items()
           if tok not in (BOLTZ / f).read_text(encoding="utf-8", errors="ignore")]
if missing:
    raise SystemExit(
        "boltz-src is NOT fully patched. Missing in:\n  " + "\n  ".join(missing) +
        f"\n\nDelete {BOLTZ} and re-run this cell."
    )
print(f"patch verified: {len(SENTINELS)}/{len(SENTINELS)} sentinels present")

## 4. Dependencies

Colab and Kaggle both ship a CUDA torch, and it is the single biggest install in
the stack -- reinstalling it costs minutes and risks pulling a build that does not
match the driver. So: keep the host's torch, install `boltz` with `--no-deps`, and
add only what the *training* path actually imports.

Three of boltz's pins are deliberately skipped, same as in the laptop's
`.venv-gpu`:

* `numpy==1.26.3` -- stale. The pipeline has been running on numpy 2.x throughout.
* `dm-tree` -- never imported under `src/boltz/`.
* `biopython` -- only `data/parse/fasta.py`, which is not on the training path.

`fairscale` is **not** skippable: it is what `activation_checkpointing: true`
actually calls.

In [ ]:
DEPS = [
    "hydra-core==1.3.2",
    "pytorch-lightning==2.4.0",
    "fairscale==0.4.13",
    "omegaconf==2.3.1",
    "einops==0.8.0",
    "mashumaro==3.14",
    "modelcif==1.8",
    "numba",          # featurizer.py imports it on the TRAINING path
    "rdkit",
    "scipy",
    "pandas",
]
run("pip install -q " + " ".join(f"'{d}'" for d in DEPS))
run(f"pip install -q --no-deps -e {BOLTZ}")

# `pip install -e` drops __editable__.boltz-1.0.0.pth into site-packages, and .pth
# files are read by site.py ONLY at interpreter startup. This kernel was already
# running, so it will not see boltz however well the install went. Boltz uses a
# src layout, so putting boltz-src/src on sys.path gets this process there
# directly -- no kernel restart, no reinstall.
import importlib, sys
BOLTZ_SRC = str(BOLTZ / "src")
if BOLTZ_SRC not in sys.path:
    sys.path.insert(0, BOLTZ_SRC)
importlib.invalidate_caches()

# boltz/__init__.py is a 6-line version shim, so importing `boltz` proves almost
# nothing. Import what train.py actually pulls in.
for mod in ["boltz.model.model", "boltz.data.module.training",
            "pytorch_lightning", "hydra", "fairscale", "rdkit", "numba"]:
    importlib.import_module(mod)
print("imports ok in this kernel")

# The kernel is not what runs training, though -- train.py runs in a subprocess.
# That is the import that actually has to work, so check it where it happens.
probe = subprocess.run(
    [sys.executable, "-c",
     "import boltz.model.model, boltz.data.module.training, fairscale, numba;"
     " import boltz; print(boltz.__file__)"],
    capture_output=True, text=True, cwd=str(BOLTZ),
)
if probe.returncode == 0:
    print(f"imports ok in a fresh subprocess: {probe.stdout.strip()}")
    PYTHONPATH_PREFIX = ""
else:
    # Editable install did not take. Hand the path to the subprocess explicitly
    # rather than trying to repair site-packages.
    print("subprocess import FAILED -- falling back to PYTHONPATH:")
    print(probe.stderr.strip()[-400:])
    PYTHONPATH_PREFIX = f"PYTHONPATH={BOLTZ_SRC} "

import torch
print("torch still:", torch.__version__, "| cuda:", torch.cuda.is_available())

# Environment for every training subprocess, in one place.
#
#  BOLTZ_CHUNK_IN_TRAINING=1  our patch. Upstream gates triangular-attention
#    chunking on `not self.training`, so training materialises the whole
#    [1, 4, 512, 512, 512] fp32 score tensor -- one 2.00 GiB allocation, and the
#    one that OOMs. Chunking computes identical math in slices of 128.
#  PYTORCH_ALLOC_CONF  the current name; PYTORCH_CUDA_ALLOC_CONF is the old one.
#    Set both, since which is honoured depends on the torch build.
ENV = (
    f"{PYTHONPATH_PREFIX}"
    "PYTHONUNBUFFERED=1 "          # child stdout is a pipe -> CPython block-buffers it
    "BOLTZ_CHUNK_IN_TRAINING=1 "
    "PYTORCH_ALLOC_CONF=expandable_segments:True "
    "PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True"
)
print("subprocess env:", ENV)

### Optional: `trifast`

This is the one genuine upside of leaving Windows. `trifast` is a Triton kernel
for triangular attention, and Boltz soft-imports it
(`triangular_attention/primitives.py:46`). It could never be installed on the
laptop -- Triton has no Windows build -- and triangular attention is precisely
where the 8 GB runs died.

It is **off by default here** because 16 GB does not need it and an unvalidated
kernel swap is a bad thing to have running underneath a result you intend to
report. To try it: install it and add
`model.pairformer_args.use_trifast=true` to the command line in section 8 --
`pairformer_args` is splatted straight into `PairformerModule`
(`model/model.py:192`), so the flag reaches the layer. Then re-run the probe in
section 7 and compare peak VRAM and s/step against the numbers you already have.
Treat it as an experiment with a before/after, not as a default.

In [ ]:
INSTALL_TRIFAST = False  # flip to True to experiment; see section 7 for the A/B

if INSTALL_TRIFAST:
    run("pip install -q trifast")
    import importlib.util
    print("trifast importable:", importlib.util.find_spec("trifast") is not None)
else:
    print("skipped (default)")

## 5. Data: the bundle, then the assets

In [ ]:
# Two shapes to handle, because Kaggle auto-extracts archives on upload: the
# dataset may hold the tarball, or it may hold the already-unpacked tree. Look
# for both rather than assuming.
def find_extracted():
    for hit in glob.glob("/kaggle/input/*/**/processed/structures", recursive=True):
        p = Path(hit)
        if any(p.glob("*.npz")):
            return p.parent.parent          # the dir containing processed/ and msa/
    return None

BUNDLE = EXTRACTED = None
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/mhc1-data-bundle.tar*", recursive=True)
    BUNDLE = hits[0] if hits else None
    if not BUNDLE:
        EXTRACTED = find_extracted()
elif ON_COLAB:
    BUNDLE = "/content/drive/MyDrive/mhc1/mhc1-data-bundle.tar.gz"  # <- edit if elsewhere

(REPO / "data").mkdir(parents=True, exist_ok=True)

def stage_link(src, dst, depth=0):
    """Merge src into dst with symlinks, WITHOUT skipping a directory that
    already exists.

    This is the subtle bit. `data/processed/` is partly tracked in git -- the
    manifests and the *_ids.txt files are, `structures/` is not -- so the clone
    leaves a data/processed/ that exists but is missing exactly the 1,084 NPZs
    we came for. Linking only when the whole directory is absent silently stages
    nothing. Descend and link the missing children instead.
    """
    dst.mkdir(parents=True, exist_ok=True)
    for child in sorted(src.iterdir()):
        target = dst / child.name
        if target.is_symlink() or target.exists():
            if child.is_dir() and target.is_dir() and not target.is_symlink():
                stage_link(child, target, depth + 1)
            continue
        try:
            target.symlink_to(child, target_is_directory=child.is_dir())
        except OSError:
            (shutil.copytree if child.is_dir() else shutil.copy2)(child, target)
        if depth == 0:
            print(f"  {target} -> {child}")

def n_npz(p):
    return len(list(p.glob("*.npz"))) if p.is_dir() else 0

if n_npz(REPO / "data/processed/structures") > 0:
    print("data already staged")

elif EXTRACTED is not None:
    # Kaggle unpacked it for us. /kaggle/input is read-only, but the training
    # path only ever reads these, so link rather than burn 570 MB of working
    # quota on a copy.
    print(f"found an extracted tree at {EXTRACTED} -- linking")
    for sub in ("processed", "msa"):
        if (EXTRACTED / sub).is_dir():
            stage_link(EXTRACTED / sub, REPO / "data" / sub)

elif BUNDLE and Path(BUNDLE).exists():
    print(f"unpacking {BUNDLE} -> {REPO}")
    t0 = time.time()
    with tarfile.open(BUNDLE) as tar:
        tar.extractall(REPO)
    print(f"done in {time.time()-t0:.0f}s")

else:
    if ON_KAGGLE:
        print("Nothing usable under /kaggle/input. What is actually there:")
        for d in sorted(glob.glob("/kaggle/input/*")):
            print(f"  {d}")
            for f in sorted(glob.glob(d + "/*"))[:10]:
                print(f"    {Path(f).name}")
    raise SystemExit(
        "Data bundle not found.\n"
        "  Build it on the laptop:  python src/make_cloud_bundle.py\n"
        "  Colab : upload to MyDrive/mhc1/ (or edit BUNDLE above)\n"
        "  Kaggle: create a Dataset from it, then Add Input on this notebook"
    )

n_struct = n_npz(REPO / "data/processed/structures")
n_msa    = n_npz(REPO / "data/msa")
print(f"\nstructures: {n_struct}   msa: {n_msa}")

if n_struct != 1084:
    print(f"\n!! expected 1084 structures, found {n_struct}. What is actually there:")
    for d in (REPO / "data", REPO / "data/processed"):
        print(f"  {d}:")
        for f in sorted(d.iterdir())[:12] if d.is_dir() else []:
            tag = "-> " + str(Path(f).resolve()) if f.is_symlink() else ("dir" if f.is_dir() else "file")
            print(f"    {f.name:<28} {tag}")
    if ON_KAGGLE:
        print("  /kaggle/input:")
        for f in sorted(glob.glob("/kaggle/input/*/*"))[:12]:
            print(f"    {f}")
    raise SystemExit("data staging incomplete -- see the listing above")

print("data staged ok")

In [ ]:
# The checkpoint (3.6 GB) and symmetry pickle (215 MB) come from their original
# hosts -- faster than pushing them through Drive, and src/fetch_assets.py already
# knows the URLs and verifies what it pulls.
assets = REPO / "data" / "assets"
if (assets / "boltz1_conf.ckpt").exists() and (assets / "symmetry.pkl").exists():
    print("assets already present")
else:
    t0 = time.time()
    run(f"cd {REPO} && python src/fetch_assets.py")
    print(f"fetched in {time.time()-t0:.0f}s")

for f in sorted(assets.iterdir()):
    print(f"  {f.name:<22} {f.stat().st_size/1024**3:.2f} GiB")

## 6. Sanity check: the pretrained baseline still reproduces

Optional, but worth doing once on a new machine. This is `validation_only` against
the 30-sample subset, and it should land near the numbers in `reports/BASELINE.md`
(~0.88 protein-protein lDDT, ~3.05 A RMSD). If it does not, the problem is the
environment, and you want to know that before spending hours training in it.

Validation is expensive here -- `sampling_steps: 200`, `diffusion_samples: 5`,
`symmetry_correction: true` -- so budget a couple of hours, or skip it and go
straight to section 7.

In [ ]:
RUN_BASELINE = False  # set True to reproduce the pretrained baseline first

if RUN_BASELINE:
    cmd = (
        f"cd {BOLTZ} && {ENV} python scripts/train/train.py "
        f"../configs/mhc1_baseline_subset30.yaml "
        f"output={RUN_DIR / 'baseline'}"
    )   # no `| tee`: the shell returns TEE's exit status, essentially always 0,
        # so a crashed or OOM'd baseline would report success. dash has no
        # pipefail. run() streams and checks the real status.
    run(cmd)
else:
    print("skipped")

## 7. Probe: does it fit, and how fast is it?

Do not start a multi-hour run on an assumption. This runs a handful of training
steps and reports two numbers:

* **peak VRAM** -- the arithmetic in `reports/GPU_REQUIREMENTS.md` predicts ~5.2 GB
  of persistent state for the frozen-trunk recipe (all weights 1.81 GB, gradients
  1.13 GB, Adam moments 2.25 GB), leaving ~10 GB for activations. The 8 GB card
  had ~2.9 GB for activations and died needing another 730 MB.
* **seconds per training batch** -- the number that decides whether the experiment
  is feasible at all, and it cannot be guessed from the 4060's timings. A T4 has
  no TF32, so fp32 matmuls run on plain CUDA cores at ~8.1 TFLOPS.

The next cell turns those into a budget for a 12 h session and a 30 h week.

In [ ]:
import re, threading

PROBE_STEPS = 8
probe_log = RUN_DIR / "probe.log"

# Training runs in a subprocess, so torch.cuda.max_memory_allocated() in THIS
# process would read 0. Poll nvidia-smi instead -- it also catches the CUDA
# context and any fragmentation, which is what actually has to fit.
peak_mib = [0]
stop = threading.Event()

def watch_vram():
    while not stop.is_set():
        try:
            out = subprocess.run(
                "nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                shell=True, capture_output=True, text=True, timeout=5)
            peak_mib[0] = max(peak_mib[0], int(out.stdout.strip().split("\n")[0]))
        except Exception:
            pass
        stop.wait(2)

cmd = (
    f"cd {BOLTZ} && {ENV} "
    f"python scripts/train/train.py ../configs/mhc1_finetune_t4.yaml "
    f"output={RUN_DIR / 'probe'} "
    # max_steps counts OPTIMIZER steps, so with accumulate_grad_batches: 16 it
    # would ask for 16x this many batches. limit_train_batches counts batches.
    f"trainer.limit_train_batches={PROBE_STEPS} trainer.max_epochs=1 "
    f"trainer.limit_val_batches=0 "
    f"disable_checkpoint=true"
)
print(f"$ {cmd}\n")

watcher = threading.Thread(target=watch_vram, daemon=True)
watcher.start()
t0 = time.time()
with open(probe_log, "w") as fh:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
        fh.write(line)
    p.wait()
elapsed = time.time() - t0
stop.set()
watcher.join(timeout=5)

print(f"\n--- probe finished in {elapsed:.0f}s (rc={p.returncode}) ---")
print(f"peak VRAM (nvidia-smi): {peak_mib[0]} MiB of {vram*1024:.0f} MiB "
      f"= {100*peak_mib[0]/(vram*1024):.0f}% of the card")

In [ ]:
# Read the probe back and turn it into a budget.
text = probe_log.read_text()

if p.returncode != 0 and "out of memory" not in text.lower():
    print(f"The probe CRASHED (exit {p.returncode}) for a non-memory reason.")
    print("Any timing below would be meaningless, so there is none. Last lines:")
    print("\n".join(text.splitlines()[-25:]))
elif "out of memory" in text.lower():
    print("OOM. It does not fit as configured. Levers, cheapest first:")
    print("  1. PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True   (already on)")
    print("  2. model.msa_args.offload_to_cpu=true, and the same for")
    print("     pairformer_args and score_model_args")
    print("  3. model.training_args.diffusion_multiplicity=8  <- changes gradient")
    print("     variance, not what is learned. Declare it.")
    print("  4. model.diffusion_loss_args.add_smooth_lddt_loss=false <- removes an")
    print("     auxiliary loss term AND the [mult, n_atoms, n_atoms] matrices the")
    print("     OOM lands in. A real objective change. Declare it.")
    print()
    print("  NOT data.max_tokens=384: tokens max out at 494, so 512 crops nothing,")
    print("  and 384 would crop 588 of 1013 records (58%) without saying so.")
else:
    # Lightning's progress bar carries the per-batch rate; fall back to wall clock.
    rates = re.findall(r"([\d.]+)\s*s/it", text)
    if rates:
        s_per_batch = float(rates[-1])
        src = "progress bar"
    else:
        s_per_batch = elapsed / PROBE_STEPS
        src = "wall clock, includes ~1-2 min startup -- pessimistic"

    ACCUM, SAMPLES = 16, 100     # accumulate_grad_batches, samples_per_epoch
    step_s  = s_per_batch * ACCUM
    epoch_h = s_per_batch * SAMPLES / 3600

    print(f"seconds per batch: {s_per_batch:.1f}   ({src})")
    print(f"per optimizer step (accum={ACCUM}): {step_s/60:.1f} min")
    print(f"per epoch ({SAMPLES} samples):      {epoch_h:.1f} h")
    print()
    print(f"one 12 h session:   ~{12/epoch_h:.1f} epochs, ~{12*3600/step_s:.0f} optimizer steps")
    print(f"a 30 h Kaggle week: ~{30/epoch_h:.1f} epochs, ~{30*3600/step_s:.0f} optimizer steps")
    print()
    print("Validation is NOT in these numbers and is expensive (sampling_steps=200,")
    print("diffusion_samples=5) -- it runs once per epoch over 30 samples.")

## 7b. Where the memory actually is: a `diffusion_multiplicity` sweep

Read the probe's `peak_gb` before running this. If it fits, skip straight to
section 8.

Three theories about this model's memory have been wrong so far, each one
reasoned from a traceback rather than measured. The measurements now available
say what is *not* the problem:

* Not fragmentation. At the peak, reserved exceeded allocated by ~0.2 GiB.
* Not the optimizer. Weights 1.61 GiB, gradients 1.04 GiB, Adam 2.09 GiB once it
  appears at batch 16 -- about 4.7 GiB of a ~13.6 GiB peak.

The rest is **`smooth_lddt_loss`** (`model/loss/diffusion.py:97`), and it is not
close. It builds `[batch x multiplicity, n_atoms, n_atoms]` fp32 matrices: at
multiplicity 16 and 4608 atoms that is **1.27 GiB each**, which is exactly the
allocation in the traceback. Roughly ten are built during mask construction and
about **ten remain live for the backward pass** -- `pred_dists`, the
`true - pred` difference, four sigmoid outputs, the broadcast `eps`, and `mask`.
That is ~10 GiB retained, on its own, before the score model is counted.

It dwarfs everything else for a simple reason: the loss works over 4608^2 atom
pairs while the trunk works over 512^2 token pairs -- an 81x larger pair
dimension.

Both levers below attack that tensor directly:

* **`max_atoms: 3904`** (already set) shrinks it ~28% at zero cost -- valid-chain
  atoms max out at 3896, so nothing is cropped.
* **`diffusion_multiplicity`** scales it linearly.

### Why the earlier 16-vs-8 comparison looked flat

It measured `max_memory_allocated` on batches that **OOM'd**. When
`training_step` catches the OOM it returns `None`, no backward runs, and the
high-water mark recorded is wherever the allocator happened to give up. At
multiplicity 16 the individual tensors are twice as large, so it fails *earlier*
in the sequence; at 8, more of the smaller tensors fit before it fails, so the
mark climbs *higher*. 13.92 > 13.61 is that artefact, not evidence against
scaling. A failure high-water mark is not a measure of demand -- the same mistake,
for the third time, which is why this cell reports OOM and crash separately from
a real peak.

`diffusion_multiplicity` is how many independently-noised copies of each
structure contribute to the diffusion loss per step. Lowering it raises gradient
variance; it does not change what the model is being asked to learn. It is a
legitimate choice under a compute constraint and **a deviation that has to be
reported** -- upstream Boltz-1 uses 16, AlphaFold-3 uses 48.

It should also make each batch meaningfully faster, which matters as much as the
memory: at 85 s/batch the wall clock is what limits this experiment, not VRAM.

In [ ]:
# Save & Run All executes every cell from the top, so a committed training run
# would repeat this ~20 minute sweep before getting to section 8. Once you know
# the answer, pin it here and the sweep is skipped.
MULTIPLICITY_OVERRIDE = None      # e.g. 8, once the table below has told you

SWEEP = [16, 8, 4, 2]
BATCHES = 3
rows = []

if MULTIPLICITY_OVERRIDE:
    MULTIPLICITY = MULTIPLICITY_OVERRIDE
    print(f"sweep skipped, using pinned diffusion_multiplicity={MULTIPLICITY}")
    SWEEP = []
    (RUN_DIR / "multiplicity.json").write_text(
        json.dumps({"diffusion_multiplicity": MULTIPLICITY, "source": "override"}))

for m in SWEEP:
    out = RUN_DIR / f"sweep_m{m}"
    shutil.rmtree(out, ignore_errors=True)
    cmd = (
        f"cd {BOLTZ} && {ENV} "
        f"python scripts/train/train.py ../configs/mhc1_finetune_t4.yaml "
        f"output={out} "
        f"model.training_args.diffusion_multiplicity={m} "
        # model.py:433 does recycling_steps = random.randint(0, training_args.
        # recycling_steps), so the number of TRUNK PASSES is random 0-3 per batch.
        # Over 3 batches that is a large uncontrolled term sitting on top of the
        # thing being measured. Pin it so the rows differ only in multiplicity.
        f"model.training_args.recycling_steps=3 "
        f"trainer.limit_train_batches={BATCHES} trainer.max_epochs=1 "
        f"trainer.limit_val_batches=0 disable_checkpoint=true"
    )
    print(f"=== diffusion_multiplicity={m} ===", flush=True)
    t0 = time.time()

    # Stream rather than capture. Loading the 3.6 GB checkpoint takes 60-90 s
    # before the first batch even starts, and a buffered subprocess.run shows
    # nothing at all until the whole run exits -- several silent minutes that
    # look identical to a hang. Echo only the lines worth seeing; the
    # state_dict warnings alone are ~200 KB per run.
    KEEP = ("[mem]", "Skipping batch", "freeze_trunk", "out of memory",
            "Error", "Traceback", "disabled:", "oom-context", "[checkpoint]")
    lines = []
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        lines.append(line)
        if any(k in line for k in KEEP):
            print("   " + line.rstrip()[:160], flush=True)
    proc.wait()
    blob = "".join(lines)
    oom = "out of memory" in blob.lower()

    peak = secs = None
    mj = out / "mem.jsonl"
    if mj.exists():
        recs = [json.loads(l) for l in mj.read_text().splitlines() if l.strip()]
        batches = [r for r in recs if r.get("event") == "batch"]
        if batches:
            peak = max(r["peak_gb"] for r in batches)
            timed = [r["secs"] for r in batches if r.get("secs")]
            secs = sum(timed) / len(timed) if timed else None
    rows.append({"mult": m, "peak": peak, "secs": secs, "oom": oom,
                 "rc": proc.returncode, "telemetry": mj.exists()})
    print(f"  peak={peak} GiB   s/batch={secs}   oom={oom}   wall={time.time()-t0:.0f}s", flush=True)
    print(flush=True)

def fmt(v, spec):
    return format(v, spec) if isinstance(v, (int, float)) else "  n/a"

def verdict(r):
    # A crashed run and an OOM'd run are different answers, and "no telemetry" is
    # not an answer at all -- reporting it as "fits: no" would turn a MemProbe
    # failure into a memory conclusion.
    if not r["telemetry"]:
        return "NO DATA"
    if r["oom"]:
        return "no"
    if r["rc"]:
        return "CRASH"
    return "yes"

print()
print(f"{'mult':>5} {'peak GiB':>9} {'s/batch':>8} {'fits':>8}")
for r in rows:
    print(f"{r['mult']:>5} {fmt(r['peak'], '9.2f')} {fmt(r['secs'], '8.1f')} "
          f"{verdict(r):>8}")
if any(not r["telemetry"] for r in rows):
    print("\nNO DATA means mem.jsonl was never written -- MemProbe failed to load,")
    print("not a memory verdict. Look for a '[mem_probe] disabled:' line above.")

ok = [r for r in rows
      if r["telemetry"] and not r["oom"] and not r["rc"]
      and r["peak"] and r["peak"] < 0.85 * vram]
if MULTIPLICITY_OVERRIDE:
    ok = []                        # nothing to report; MULTIPLICITY already set
if ok:
    best = max(ok, key=lambda r: r["mult"])        # largest multiplicity that fits
    print()
    print(f"Largest multiplicity that fits with headroom: {best['mult']} "
          f"(peak {best['peak']:.2f} of {vram:.1f} GiB, {best['secs']:.0f} s/batch)")
    print(f"Set it in section 8 with:")
    print(f"  model.training_args.diffusion_multiplicity={best['mult']}")
    if best["secs"]:
        print()
        print(f"At {best['secs']:.0f} s/batch: "
              f"{best['secs']*16/60:.0f} min per optimizer step, "
              f"{best['secs']*100/3600:.1f} h per epoch, "
              f"~{12*3600/(best['secs']*16):.0f} steps in a 12 h session.")
    MULTIPLICITY = best["mult"]
    # Persist it: section 8 is what you re-run after a disconnect, and re-running
    # sections 1-5 does not re-bind this name.
    (RUN_DIR / "multiplicity.json").write_text(
        json.dumps({"diffusion_multiplicity": MULTIPLICITY,
                    "peak_gb": best["peak"], "secs_per_batch": best["secs"]}))
    print(f"wrote {RUN_DIR / 'multiplicity.json'}")
elif not MULTIPLICITY_OVERRIDE:
    print()
    print("Nothing fits. Next lever is data.max_tokens=384, which changes the crop "
          "and therefore the objective -- report it if you use it.")
    MULTIPLICITY = None

## 8. The run

`max_epochs: -1` plus a 12 h wall means this run will be interrupted. That is
fine, and it is designed for: the checkpoint callback writes `last.ckpt` into
`RUN_DIR`, and the cell below passes it back as `resume=` if it is there.

When `resume` is set, `pretrained` is ignored (`scripts/train/train.py:130`),
which is the correct behaviour -- you want the optimizer state back, not a fresh
load of the pretrained weights.

### Leaving it running

**Colab**: the tab has to stay open. `RUN_DIR` is Drive, so `last.ckpt` is simply
still there next time; re-run sections 1-5 and then this cell.

**Kaggle**: use **Save Version -> Save & Run All (Commit)**. The job runs
headless and survives the tab closing, which is the main reason to prefer Kaggle.
Two things to get right first:

1. **Pin the multiplicity.** Save & Run All executes every cell from the top, so
   an unpinned run repeats the ~20 minute sweep in 7b before reaching this cell.
   Set `MULTIPLICITY_OVERRIDE` there once you know the answer.
2. **Turn on "Always save output"** in the version dialog, or `RUN_DIR` is
   discarded when the session ends.

### Resuming on Kaggle is not just re-running

This is the part that differs from Colab and is easy to get wrong.
`/kaggle/working` starts **empty in every session** -- it is not shared storage,
it is the staging area for one version's output. So `last.ckpt` from your last
12 h run is not sitting there waiting.

To continue:

1. Open the finished version and confirm `runs/t4/last.ckpt` is in its **Output**.
2. In a new version, **+ Add Input -> Your Work -> Notebooks**, and attach that
   notebook's output.
3. Run as normal. The cell below searches `/kaggle/input/**/last.ckpt` and picks
   it up automatically.

Each 12 h session is therefore one link in a chain, and the chain is manual. At
~32 optimizer steps per session that is worth knowing before you plan a week
around it.

In [ ]:
# The patched train.py pins ModelCheckpoint's dirpath to `output`, so last.ckpt
# is at RUN_DIR/last.ckpt. Unpatched Boltz puts it in RUN_DIR/checkpoints/ instead
# (no dirpath + no logger -> default_root_dir/"checkpoints"), so search both --
# this cell should keep working against an older run directory.
candidates = [p for p in RUN_DIR.rglob("last.ckpt") if p.is_file()]

# On Colab, RUN_DIR is Drive and last.ckpt is simply still there. On Kaggle only
# an INTERACTIVE session carries /kaggle/working over (Notebook options ->
# Persistence -> Files only); a committed run starts empty and the previous
# version's output has to be attached as an input, mounting under /kaggle/input.
if not candidates and ON_KAGGLE:
    candidates = [Path(p) for p in glob.glob("/kaggle/input/**/last.ckpt",
                                             recursive=True)]

# Newest, not lexicographically first -- with several previous outputs attached,
# sorted()[0] resumes from the OLDEST link in the chain.
resume = max(candidates, key=lambda p: p.stat().st_mtime) if candidates else None
if len(candidates) > 1:
    print(f"{len(candidates)} checkpoints found; taking the newest")

train_log = RUN_DIR / f"finetune_{time.strftime('%Y%m%d_%H%M%S')}.log"

parts = [
    f"cd {BOLTZ}",
    f"&& {ENV}",
    "python scripts/train/train.py ../configs/mhc1_finetune_t4.yaml",
    f"output={RUN_DIR}",
]
# Carry the sweep's choice through. It is written to disk by 7b because this cell
# is the one you re-run after a disconnect, and the markdown says to re-run
# sections 1-5 -- which does NOT include 7b. Relying on the in-memory name meant
# that after every reconnect the run silently fell back to the config default of
# 16, the exact value 7b exists to replace, and the one that OOMs.
_mult = globals().get("MULTIPLICITY")
_mult_file = RUN_DIR / "multiplicity.json"
if not _mult and _mult_file.exists():
    _mult = json.loads(_mult_file.read_text()).get("diffusion_multiplicity")
    print(f"diffusion_multiplicity={_mult} (recovered from {_mult_file.name})")

if _mult:
    parts.append(f"model.training_args.diffusion_multiplicity={_mult}")
    print(f"diffusion_multiplicity={_mult} -- REPORT THIS, upstream is 16")
else:
    print("!! no MULTIPLICITY set -- using the config default (16), which OOMs on")
    print("!! a 16 GB T4. Run section 7b, or set MULTIPLICITY_OVERRIDE there.")

if resume is not None:
    parts.append(f"resume={resume}")
    print(f"RESUMING from {resume} ({resume.stat().st_size/1024**3:.1f} GiB)")
else:
    # Say this loudly. Silently restarting from pretrained weights looks exactly
    # like a working resume until you notice the loss curve begins again.
    print("!! NO CHECKPOINT FOUND -- starting fresh from the pretrained weights.")
    print("!! If you expected to resume, stop and check:")
    print(f"!!   ls -R {RUN_DIR}")
    if ON_KAGGLE:
        print("!!   and that the previous version's output is attached as an Input")

# Give the trainer the time that is actually left, minus a reserve for the final
# checkpoint write (~4 GB) and the output upload (~8 GB).
RESERVE_H = 1.0
_used_h = (time.time() - SESSION_START) / 3600
_left_h = max(0.25, SESSION_CAP_H - _used_h - RESERVE_H)
_h, _m = int(_left_h), int((_left_h % 1) * 60)
parts.append(f'trainer.max_time="00:{_h:02d}:{_m:02d}:00"')
print(f"session used {_used_h:.1f} h of {SESSION_CAP_H:.0f}; "
      f"trainer gets {_h}h{_m:02d}m, {RESERVE_H:.0f} h reserved for saving")
if _left_h < 1.0:
    print("!! under an hour of usable time left -- consider a fresh session")
    print("!! and resuming, rather than training for a few minutes.")

cmd = " ".join(parts)
print(f"$ {cmd}\nlogging to {train_log}\n")

with open(train_log, "w") as fh:
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            print(line, end="")
            fh.write(line)
            fh.flush()          # so the log survives a hard session kill
    except KeyboardInterrupt:
        p.terminate()
        print("\ninterrupted -- last.ckpt in RUN_DIR is the resume point")
    p.wait()
print(f"\nexit {p.returncode}")
if p.returncode:
    # Under Kaggle "Save & Run All" a run that died in minute three
    # would otherwise commit as a perfectly healthy version.
    raise SystemExit(f"training failed (exit {p.returncode}) -- see {train_log}")

## 9. What came out

`ValProgressDump` (our patch, `src/val_progress.py`) snapshots the running
validation metrics to `val_state.json` after every batch, so an interrupted
validation is not a total loss -- which matters a lot more on a 12 h session than
it did on the laptop.

In [ ]:
vs = RUN_DIR / "val_state.json"
if vs.exists():
    print(json.dumps(json.loads(vs.read_text()), indent=2)[:2000])
else:
    print("no val_state.json yet -- validation has not run")

print("\ncheckpoints:")
for f in sorted(RUN_DIR.glob("*.ckpt")):
    print(f"  {f.name:<40} {f.stat().st_size/1024**3:.2f} GiB")

In [ ]:
# Copy the run artefacts back into the repo tree so they can be committed from the
# laptop. Checkpoints are deliberately NOT copied -- they are ~4 GB each.
# REPO is /kaggle/tmp on Kaggle -- scratch. Copying there persists nothing.
dest = (RUN_DIR / "artifacts") if ON_KAGGLE else (REPO / "reports" / "t4")
dest.mkdir(parents=True, exist_ok=True)
seen = set()
for pat in ("*.log", "*.json", "*.jsonl"):     # *.jsonl = mem.jsonl, the telemetry
    for f in sorted(RUN_DIR.rglob(pat)):       # rglob: sweep_m*/ and probe/ too
        if f.is_file() and dest not in f.parents and f.name not in seen:
            seen.add(f.name)
            shutil.copy2(f, dest / f.name)
            print("copied", f.name)

print(f"\nDownload {dest} and commit it. On Kaggle everything under "
      f"/kaggle/working is already in the committed version's Output tab.")